# ScholarAI Flask Integration Notebook
## Use the trained model inside your Flask website

This notebook shows exactly how to use the exported `joblib` model in your Flask app so that the **Predicted Score** column on:

`http://127.0.0.1:5000/admin/predictions`

comes from a trained ML model instead of the old weighted formula.

## 1. Upload the exported artifacts to Colab (or keep them in your project)

Expected files:
- `scholarai_academic_score_pipeline.joblib`
- `scholarai_model_metadata.json`
- `scholarai_business_rules.json`

In the Flask project, place them inside:

```text
scholarai/app/artifacts/
```

In [ ]:

import json
from pathlib import Path

import joblib
import pandas as pd

In [ ]:

ARTIFACT_DIR = Path("/content/scholarai_artifacts")  # change if needed
MODEL_PATH = ARTIFACT_DIR / "scholarai_academic_score_pipeline.joblib"
META_PATH = ARTIFACT_DIR / "scholarai_model_metadata.json"
RULES_PATH = ARTIFACT_DIR / "scholarai_business_rules.json"

model = joblib.load(MODEL_PATH)
with open(META_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)
with open(RULES_PATH, "r", encoding="utf-8") as f:
    rules = json.load(f)

metadata

## 2. Shared Flask-compatible helper functions

Use one shared file in Flask, for example:

`app/ml_service.py`

Both `admin.py` and `student.py` should import from this shared file so the prediction logic stays identical.

In [ ]:
ML_SERVICE_CODE = 'import json\nimport joblib\nimport pandas as pd\nfrom pathlib import Path\n\nARTIFACT_DIR = Path(__file__).resolve().parent / "artifacts"\nMODEL_PATH = ARTIFACT_DIR / "scholarai_academic_score_pipeline.joblib"\nMETA_PATH = ARTIFACT_DIR / "scholarai_model_metadata.json"\nRULES_PATH = ARTIFACT_DIR / "scholarai_business_rules.json"\n\n_model = None\n_metadata = None\n_rules = None\n\ndef _load_artifacts():\n    global _model, _metadata, _rules\n    if _model is None:\n        _model = joblib.load(MODEL_PATH)\n    if _metadata is None:\n        with open(META_PATH, "r", encoding="utf-8") as f:\n            _metadata = json.load(f)\n    if _rules is None:\n        with open(RULES_PATH, "r", encoding="utf-8") as f:\n            _rules = json.load(f)\n    return _model, _metadata, _rules\n\ndef score_to_risk(score):\n    if score >= 75:\n        return "low"\n    elif score >= 55:\n        return "medium"\n    return "high"\n\ndef calculate_trend(term1_score, term2_score, term3_score, audience="admin"):\n    diff1 = term2_score - term1_score\n    diff2 = term3_score - term2_score\n\n    if audience == "student":\n        if diff1 > 3 and diff2 > 3:\n            return "improving", "↑ Improving", "You are consistently improving each term. Keep it up!"\n        elif diff1 < -3 and diff2 < -3:\n            return "declining", "↓ Declining", "Your marks are dropping each term. Seek help immediately."\n        elif diff1 > 3 and diff2 < -3:\n            return "unstable", "~ Unstable", "Marks went up then dropped. Try to maintain consistency."\n        elif diff1 < -3 and diff2 > 3:\n            return "recovering", "↑ Recovering", "Marks dropped in term 2 but you recovered in term 3. Well done!"\n        else:\n            return "stable", "→ Stable", "Performance is consistent across all terms."\n    else:\n        if diff1 > 3 and diff2 > 3:\n            return "improving", "↑ Improving", "Student is consistently improving each term."\n        elif diff1 < -3 and diff2 < -3:\n            return "declining", "↓ Declining", "Student marks are dropping each term. Immediate attention needed."\n        elif diff1 > 3 and diff2 < -3:\n            return "unstable", "~ Unstable", "Marks went up then dropped. Performance is inconsistent."\n        elif diff1 < -3 and diff2 > 3:\n            return "recovering", "↑ Recovering", "Marks dropped in term 2 but recovered in term 3. Monitor closely."\n        else:\n            return "stable", "→ Stable", "Performance is consistent across all terms."\n\ndef predict_academic_score(attendance_rate, term1_score, term2_score, term3_score):\n    model, _, _ = _load_artifacts()\n    terminal_avg = (term1_score + term2_score + term3_score) / 3\n\n    input_df = pd.DataFrame([{\n        "attendance_rate": attendance_rate,\n        "term1_score": term1_score,\n        "term2_score": term2_score,\n        "term3_score": term3_score,\n        "terminal_avg": terminal_avg\n    }])\n\n    predicted_score = float(model.predict(input_df)[0])\n    return round(max(0, min(100, predicted_score)), 2)\n\ndef calculate_final_risk(predicted_score, attendance_rate, complaint_count, due_amount, trend):\n    base_risk = score_to_risk(predicted_score)\n    risk_flags = []\n\n    if complaint_count >= 3:\n        risk_flags.append("HIGH BEHAVIOR RISK")\n    elif complaint_count >= 1:\n        risk_flags.append("BEHAVIOR WARNING")\n\n    if due_amount > 5000:\n        risk_flags.append("HIGH FINANCIAL RISK")\n    elif due_amount > 0:\n        risk_flags.append("FINANCIAL WARNING")\n\n    if attendance_rate < 65:\n        risk_flags.append("CRITICAL ATTENDANCE")\n    elif attendance_rate < 75:\n        risk_flags.append("LOW ATTENDANCE")\n\n    if predicted_score < 55:\n        risk_flags.append("POOR ACADEMIC PERFORMANCE")\n\n    if trend == "declining":\n        risk_flags.append("DECLINING TREND")\n    elif trend == "unstable":\n        risk_flags.append("UNSTABLE TREND")\n\n    high_flags = {"HIGH BEHAVIOR RISK", "HIGH FINANCIAL RISK", "CRITICAL ATTENDANCE", "DECLINING TREND"}\n\n    if base_risk == "low" and (len(risk_flags) >= 2 or any(f in high_flags for f in risk_flags)):\n        final_risk = "medium"\n    elif base_risk == "medium" and len(risk_flags) >= 1:\n        final_risk = "high"\n    else:\n        final_risk = base_risk\n\n    return final_risk, risk_flags\n\ndef get_grade(score):\n    if score >= 85:\n        return "A"\n    elif score >= 75:\n        return "B+"\n    elif score >= 65:\n        return "B"\n    elif score >= 55:\n        return "C+"\n    elif score >= 45:\n        return "C"\n    return "F"\n\ndef get_pi_label(score):\n    if score >= 85:\n        return "Excellent"\n    elif score >= 75:\n        return "Good"\n    elif score >= 55:\n        return "Average"\n    return "Below Average"'
print(ML_SERVICE_CODE[:2500])

In [ ]:

from pathlib import Path
helper_path = Path("/content/ml_service.py")
helper_path.write_text(ML_SERVICE_CODE, encoding="utf-8")
print("Saved helper module to:", helper_path)

## 3. Admin route patch

In your current `admin.py`, replace the manual formula with the trained model.

In [ ]:
ADMIN_PATCH = r"""from app.ml_service import (
    predict_academic_score,
    calculate_trend,
    calculate_final_risk,
    get_grade,
    get_pi_label,
)

# Inside run_prediction():
predicted_score = predict_academic_score(
    attendance_rate=attendance_rate,
    term1_score=term1_score,
    term2_score=term2_score,
    term3_score=term3_score,
)

comp_row = fetch_one(
    "SELECT NVL(complaint_count, 0) AS cnt, NVL(due_amount, 0) AS due_amt FROM students WHERE student_id = :sid",
    {"sid": student_id}
)
complaint_count = comp_row["cnt"] if comp_row else 0
due_amount_live = comp_row["due_amt"] if comp_row else due_amount

trend, trend_label, trend_note = calculate_trend(term1_score, term2_score, term3_score, audience="admin")
final_risk, risk_flags = calculate_final_risk(
    predicted_score=predicted_score,
    attendance_rate=attendance_rate,
    complaint_count=complaint_count,
    due_amount=due_amount_live,
    trend=trend,
)

grade = get_grade(predicted_score)
pi_label = get_pi_label(predicted_score)"""
print(ADMIN_PATCH)

## 4. Student route patch

Do the same in `student.py`, so the student portal uses the **same trained model and same risk logic** as the admin side.

In [ ]:
STUDENT_PATCH = 'from app.ml_service import (\n    predict_academic_score,\n    calculate_trend,\n    calculate_final_risk,\n    get_grade,\n    get_pi_label,\n)\n\n# Inside predict():\nstudent_meta = fetch_one(\n    """\n    SELECT\n        NVL(complaint_count, 0) AS complaint_count,\n        NVL(due_amount, 0) AS due_amount\n    FROM students\n    WHERE student_id = :sid\n    """,\n    {"sid": sid}\n) or {"complaint_count": 0, "due_amount": 0}\n\npredicted_score = predict_academic_score(\n    attendance_rate=attendance_rate,\n    term1_score=term1_score,\n    term2_score=term2_score,\n    term3_score=term3_score\n)\n\ntrend, trend_label, trend_note = calculate_trend(\n    term1_score, term2_score, term3_score, audience="student"\n)\n\nrisk_level, risk_flags = calculate_final_risk(\n    predicted_score=predicted_score,\n    attendance_rate=attendance_rate,\n    complaint_count=int(student_meta["complaint_count"]),\n    due_amount=float(student_meta["due_amount"]),\n    trend=trend\n)\n\ngrade = get_grade(predicted_score)\npi_label = get_pi_label(predicted_score)'
print(STUDENT_PATCH)

## 5. Predicted Score column behavior

In the Admin Predictions page, keep using:

```html
<td><span class="mono fw-800 {{ text_color }}">{{ p.predicted_score }}%</span></td>
```

That column does **not** need HTML changes.
It will automatically show the ML prediction once Flask saves the new `predicted_score` value into the database.

In [ ]:

def quick_demo(attendance_rate, term1_score, term2_score, term3_score):
    terminal_avg = (term1_score + term2_score + term3_score) / 3
    x = pd.DataFrame([{
        "attendance_rate": attendance_rate,
        "term1_score": term1_score,
        "term2_score": term2_score,
        "term3_score": term3_score,
        "terminal_avg": terminal_avg
    }])
    predicted_score = float(model.predict(x)[0])
    predicted_score = round(max(0, min(100, predicted_score)), 2)

    if predicted_score >= 75:
        risk = "low"
    elif predicted_score >= 55:
        risk = "medium"
    else:
        risk = "high"

    return {
        "predicted_score": predicted_score,
        "risk_level_from_score": risk,
        "attendance_rate": attendance_rate,
        "term1_score": term1_score,
        "term2_score": term2_score,
        "term3_score": term3_score
    }

quick_demo(82, 71, 68, 74)

## 6. Final Flask project checklist

1. Put artifacts inside `app/artifacts/`
2. Add dependencies to `requirements.txt`:
   - `pandas`
   - `numpy`
   - `scikit-learn`
   - `joblib`
3. Create `app/ml_service.py`
4. Update **both** `admin.py` and `student.py`
5. Run prediction from the form
6. Check `admin/predictions` → the **Predicted Score** column will now show ML-generated values